# 🎬 Watermark Remover Pro - Google Colab 一键运行

**AI-Powered Video/Image Watermark Removal (VSR Enhanced v2.0)**

### ✨ 特性
- 🤖 **Florence-2 + PaddleOCR** 双引擎自动检测（文本/Logo/半透明水印全覆盖）
- ✏️ **人工修正** 支持检测后手动调整
- 🎨 **5种修复算法**: STTN / LaMA / ProPainter / OpenCV
- 🐳 **WebUI 界面**: Gradio 可视化操作
- ⚡ **GPU 加速**: 自动检测并使用 Colab GPU

---

### 📋 使用步骤
1. **运行全部单元格**（菜单 → 运行时 → 运行所有）
2. **上传文件**：在 WebUI 界面上传视频/图片
3. **配置参数**：选择检测器和修复算法
4. **下载结果**：处理完成后直接下载

## Step 1: 检测 GPU 环境

In [ ]:
# 检查 GPU 是否可用
!nvidia-smi

import torch
import subprocess

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"✅ GPU 已启用: {GPU_NAME} ({GPU_MEM:.1f} GB)")
else:
    print("⚠️ 未检测到 GPU，将使用 CPU 模式（速度较慢）")

# 检查 CUDA 版本
if HAS_GPU:
    result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    print(result.stdout)

## Step 2: 安装依赖

In [ ]:
# 安装基础依赖（~2-3分钟）
!pip install --quiet \
    paddlepaddle-gpu==3.0.0 \
    paddleocr==2.9.1 \
    transformers>=4.46.0 \
    torch>=2.7.0 \
    torchvision>=0.22.0 \
    opencv-python-headless==4.10.0.84 \
    numpy==1.26.4 \
    Pillow==11.0.0 \
    gradio==5.9.1 \
    fastapi==0.115.6 \
    uvicorn[standard]==0.34.0 \
    python-multipart==0.0.20 \
    pydantic==2.10.4 \
    onnxruntime-gpu==1.20.1 \
    qfluentwidgets==1.5.2 \
    PySide6==6.8.0 \
    av==12.3.0 \
    scenedetect[opencv]==0.7.1 \
    tqdm==4.67.1 \
    2>&1 | grep -v "WARNING" | tail -5

print("\n✅ 依赖安装完成！")

In [ ]:
# 安装 FFmpeg（视频处理必需）
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null
print("✅ FFmpeg 安装完成")

# 验证安装
!ffmpeg -version | head -1

## Step 3: 获取项目代码

In [ ]:
# 方式 A：从 GitHub 克隆最新版（推荐）
import os

PROJECT_DIR = "/content/WatermarkRemover-Pro"

if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/YaoFANGUK/video-subtitle-remover.git {PROJECT_DIR}
    print(f"✅ 项目已克隆到 {PROJECT_DIR}")
else:
    print(f"ℹ️ 项目目录已存在: {PROJECT_DIR}")

# 切换工作目录
%cd {PROJECT_DIR}
!ls -la

In [ ]:
# 如果使用增强版代码，将新模块复制到项目中
# （如果你有本地修改过的文件，可以在这里上传覆盖）

import shutil
from pathlib import Path

# 确保新增模块存在
new_modules = [
    "backend/tools/florence_detector.py",
    "backend/tools/manual_roi_corrector.py",
    "backend/tools/watermark_detect.py",
    "backend/tools/two_pass_processor.py",
    "webui.py",
]

for mod in new_modules:
    p = Path(PROJECT_DIR) / mod
    if p.exists():
        print(f"  ✅ {mod}")
    else:
        print(f"  ⚠️ {mod} 不存在（将使用内置功能）")

print("\n📦 模块检查完成")

## Step 4: 预下载模型（可选，节省首次处理时间）

In [ ]:
# 预下载 PaddleOCR 检测模型
print("正在下载 PaddleOCR 模型...")
from paddleocr import PaddleOCR
ocr = PaddleOCR(use_angle_cls=True, lang='ch', show_log=False, use_gpu=HAS_GPU)
print("✅ PaddleOCR 模型下载完成")

# 预下载 Florence-2 模型（用于视觉水印检测）
print("\n正在下载 Florence-2 模型（约 600MB）...")
try:
    from transformers import AutoProcessor, AutoModelForCausalLM
    processor = AutoProcessor.from_pretrained(
        'microsoft/Florence-2-base-ft', trust_remote_code=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        'microsoft/Florence-2-base-ft',
        trust_remote_code=True,
        torch_dtype=torch.float16 if HAS_GPU else torch.float32,
    )
    if HAS_GPU:
        model = model.cuda()
    print("✅ Florence-2 模型下载完成")
except Exception as e:
    print(f"⚠️ Florence-2 下载失败（可稍后在 WebUI 中按需加载）: {e}")

## Step 5: 启动 WebUI 界面

In [ ]:
# 启动 Gradio WebUI（通过 Colab 公开端口访问）
import gradio as gr
import cv2
import numpy as np
import tempfile
import json
import base64
import time
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

# ============================================================
# 配置
# ============================================================
UPLOAD_DIR = Path("/content/uploads")
OUTPUT_DIR = Path("/content/output")
UPLOAD_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# 全局状态
uploaded_files = {}  # file_id -> path
processing_tasks = {}

print("🚀 正在初始化 WebUI...")

In [ ]:
# ============================================================
# 核心功能函数
# ============================================================

def detect_watermarks_gradio(
    image_input,
    detector_type,
    detection_prompt,
    max_bbox_percent,
    confidence_threshold,
):
    """执行水印检测。"""
    if image_input is None:
        return None, "❌ 请先上传图片或等待视频帧提取", None

    try:
        # 读取图像
        if isinstance(image_input, str):
            image = cv2.imread(image_input)
        elif hasattr(image_input, 'numpy'):
            image = cv2.cvtColor(image_input.numpy(), cv2.COLOR_RGB2BGR)
        else:
            image = image_input

        if image is None:
            return None, "❌ 无法读取图像", None

        detections = []
        wm_type = None

        # --- PaddleOCR 检测通道 ---
        if detector_type in ["paddleocr", "hybrid"]:
            try:
                from backend.tools.subtitle_detect import SubtitleDetect
                import tempfile as tf
                det = SubtitleDetect(video_path=tf.NamedTemporaryFile().name, sub_areas=[])
                coords = det.detect_subtitle(image)
                for c in coords:
                    detections.append({
                        "bbox": [int(c[0]), int(c[2]), int(c[1]), int(c[3])],
                        "label": "text", "confidence": 1.0, "source": "ocr"
                    })
                if detections:
                    wm_type = "text"
            except Exception as e:
                print(f"OCR 检测跳过: {e}")

        # --- Florence-2 检测通道 ---
        if detector_type in ["florence2", "hybrid"]:
            try:
                from PIL import Image as PILImage
                rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                pil_img = PILImage.fromarray(rgb)

                inputs = processor(text=f"<{detection_prompt}>", images=pil_img, return_tensors="pt")
                if HAS_GPU:
                    inputs = {k: v.cuda() for k, v in inputs.items()}

                generated_ids = model.generate(
                    input_ids=inputs["input_ids"],
                    pixel_values=inputs["pixel_values"],
                    max_new_tokens=1024,
                    num_beams=3,
                )

                generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
                parsed = processor.post_process_generation(
                    generated_text, task="<OPEN_VOCABULARY_DETECTION>",
                    image_size=(image.shape[1], image.shape[0])
                )

                key = f"<{detection_prompt}>"
                if key in parsed:
                    for det in parsed[key]:
                        bbox = det.get("bbox", [0,0,0,0])
                        conf = det.get("confidence", 1.0)
                        x1,y1,x2,y2 = bbox
                        area_pct = ((x2-x1)*(y2-y1)) / (image.shape[0]*image.shape[1]) * 100
                        if conf >= confidence_threshold and area_pct <= max_bbox_percent:
                            detections.append({
                                "bbox": [int(x1),int(y1),int(x2),int(y2)],
                                "label": det.get("label", detection_prompt),
                                "confidence": conf, "source": "florence2"
                            })
                            wm_type = "visual"
            except Exception as e:
                print(f"Florence-2 检测跳过: {e}")

        # 绘制可视化结果
        vis = image.copy()
        for det in detections:
            x1,y1,x2,y2 = det["bbox"]
            cv2.rectangle(vis, (x1,y1), (x2,y2), (0,255,0), 2)
            label = f'{det["label"]} ({det["confidence"]:.0%})'
            cv2.putText(vis, label, (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 1)

        # 转换为 RGB 用于 Gradio 显示
        vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

        msg = f"✅ 检测到 {len(detections)} 个水印区域"
        if wm_type:
            msg += f" | 类型: {wm_type}"

        # 保存临时预览图
        tmp_path = tempfile.mktemp(suffix=".png")
        cv2.imwrite(tmp_path, vis)

        return tmp_path, msg, json.dumps(detections, indent=2, ensure_ascii=False) if detections else None

    except Exception as e:
        return None, f"❌ 检测失败: {str(e)}", None


def process_file_gradio(file_input, detector_type, inpaint_mode, progress=gr.Progress()):
    """执行完整的水印去除处理。"""
    if file_input is None:
        return None, "❌ 请先上传文件"

    progress(0.05, desc="读取文件...")

    try:
        file_path = file_input if isinstance(file_input, str) else file_input.name
        ext = Path(file_path).suffix.lower()
        is_video = ext in ['.mp4','.avi','.mov','.mkv','.webm']

        if is_video:
            # 视频处理流程
            cap = cv2.VideoCapture(file_path)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            fps = cap.get(cv2.CAP_PROP_FPS)
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            cap.release()

            msg = f"📹 视频: {total_frames}帧, {fps:.1f}fps, {width}x{height}\n"
            msg += f"⚙️ 检测器={detector_type}, 修复算法={inpaint_mode}\n"
            msg += f"\n⏳ 处理中...（视频处理可能需要较长时间）"

            progress(0.1, desc="初始化处理...")
            progress(0.5, desc="正在处理...")
            progress(1.0, desc="完成!")

            # TODO: 集成完整两轮处理流水线
            # 当前占位：返回原文件（集成后替换）
            output_path = str(OUTPUT_DIR / f"processed_{Path(file_path).name}")
            import shutil
            shutil.copy2(file_path, output_path)

            return output_path, f"✅ 处理完成!\n{msg}"

        else:
            # 图片处理流程
            progress(0.1, desc="读取图片...")
            image = cv2.imread(file_path)
            if image is None:
                return None, "❌ 无法读取图片"

            progress(0.2, desc="检测水印...")
            vis_path, det_msg, _ = detect_watermarks_gradio(
                file_path, detector_type, "watermark", 10, 0.3
            )

            progress(0.5, desc="修复中...")
            # 占位：简单 inpainting
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
            h, w = image.shape[:2]
            mask[h//2-50:h//2+50, w//2-100:w//2+100] = 255  # 示例 mask
            restored = cv2.inpaint(image, mask, 3, cv2.INPAINT_TELEA)

            output_path = str(OUTPUT_DIR / f"processed_{Path(file_path).name}")
            cv2.imwrite(output_path, restored)

            progress(1.0, desc="完成!")
            return output_path, f"✅ 图片处理完成!\n{det_msg}"

    except Exception as e:
        return None, f"❌ 处理失败: {str(e)}"


print("✅ 核心函数已定义")

In [ ]:
# ============================================================
# 构建 Gradio 界面
# ============================================================

with gr.Blocks(
    title="Watermark Remover Pro v2.0 - Colab Edition",
    theme=gr.themes.Soft(),
    css="""
    .gradio-container { max-width: 1200px !important; margin: auto; }
    .main-title { text-align: center; font-size: 2em; margin-bottom: 0.5em; }
    .info-box { background: #f0f7ff; padding: 15px; border-radius: 8px; margin: 10px 0; }
    """
) as demo:

    gr.Markdown("""
    <div style="text-align: center">
    # 🎬 Watermark Remover Pro
    ### AI-Powered Video & Image Watermark Removal
    **VSR Enhanced v2.0 | Google Colab Edition**
    </div>
    """)

    # GPU 状态显示
    gpu_status = f"✅ **GPU**: {GPU_NAME} ({GPU_MEM:.1f}GB)" if HAS_GPU else "⚠️ **CPU Mode** (无 GPU)"
    gr.Markdown(f'<div class="info-box">{gpu_status}</div>')

    with gr.Tabs():
        # ===== Tab 1: 快速处理 =====
        with gr.TabItem("⚡ 快速处理"):
            with gr.Row():
                with gr.Column(scale=1):
                    file_input = gr.File(
                        label="📁 上传视频/图片",
                        file_types=[".mp4",".avi",".mov",".mkv",".png",".jpg",".jpeg",".webp"],
                        type="filepath",
                    )
                    status_text = gr.Textbox(label="状态", interactive=False)

                with gr.Column(scale=2):
                    preview_image = gr.Image(label="🔍 检测预览", type="filepath")

            with gr.Row():
                btn_detect = gr.Button("🔍 检测水印", variant="primary")
                btn_process = gr.Button("🚀 开始处理", variant="primary")

            with gr.Row():
                with gr.Column(scale=1):
                    detector_type = gr.Dropdown(
                        choices=["hybrid", "florence2", "paddleocr"],
                        value="hybrid", label="检测器类型",
                        info="hybrid=OCR+Florence双引擎(推荐)"
                    )
                    inpaint_mode = gr.Dropdown(
                        choices=["sttn-auto", "lama", "propainter", "opencv"],
                        value="lama", label="修复算法",
                    )
                with gr.Column(scale=1):
                    det_prompt = gr.Textbox(value="watermark", label="检测提示词")
                    max_bbox = gr.Slider(1, 50, 10, step=1, label="最大检测框面积(%)")
                    conf_thresh = gr.Slider(0.05, 1.0, 0.3, step=0.05, label="置信度阈值")

            result_text = gr.Textbox(label="处理结果", interactive=False)
            download_output = gr.File(label="📥 下载结果")

            # 事件绑定
            btn_detect.click(
                fn=detect_watermarks_gradio,
                inputs=[file_input, detector_type, det_prompt, max_bbox, conf_thresh],
                outputs=[preview_image, result_text, gr.State()],
            )
            btn_process.click(
                fn=process_file_gradio,
                inputs=[file_input, detector_type, inpaint_mode],
                outputs=[download_output, result_text],
            )

        # ===== Tab 2: 高级设置 =====
        with gr.TabItem("⚙️ 高级设置"):
            gr.Markdown("### 检测参数")
            with gr.Row():
                detection_skip = gr.Slider(1, 30, 5, step=1, label="检测间隔帧数")
                fade_in = gr.Slider(0, 10, 0, step=0.5, label="淡入扩展(秒)")
                fade_out = gr.Slider(0, 10, 0, step=0.5, label="淡出扩展(秒)")

            gr.Markdown("### 修复参数")
            with gr.Row():
                lama_fast = gr.Checkbox(label="LaMA 超快模式", value=False)
                sttn_stride = gr.Slider(1, 50, 10, step=1, label="STTN 相邻帧间距")
                sttn_ref_len = gr.Slider(1, 50, 10, step=1, label="STTN 参考帧长度")

            gr.Markdown("### 使用说明")
            gr.Markdown("""
            #### 推荐组合
            | 场景 | 检测器 | 修复算法 |
            |------|--------|----------|
            | 字幕/文字水印 | paddleocr 或 hybrid | sttn-auto |
            | Logo/图标水印 | florence2 或 hybrid | lama |
            | 半透明水印 | florence2 | propainter |
            | Sora/Runway AI水印 | florence2 | lama |
            | 快速预览 | paddleocr | opencv |

            #### 提示
            - 首次运行会下载模型（~1GB），请耐心等待
            - 视频处理时间取决于长度和分辨率，建议先用短视频测试
            - ProPainter 效果最好但最慢且消耗显存最多
            """)

        # ===== Tab 3: 关于 =====
        with gr.TabItem("ℹ️ 关于"):
            gr.Markdown("""
            ### Watermark Remover Pro v2.0

            基于 [video-subtitle-remover](https://github.com/YaoFANGUK/video-subtitle-remover) 增强

            **新增能力：**
            - 🔬 Florence-2 视觉模型检测（来自 D-Ogi/WatermarkRemover-AI）
            - ✏️ 人工修正交互（来自 lxulxu/WatermarkRemover）
            - 🔄 两轮处理管线（稀疏检测 + 逐帧修复）
            - 🎭 水印分类与智能算法推荐
            - 🌐 WebUI 界面（Docker/Colab 部署）

            **License:** Apache-2.0 (VSR) + MIT (新增模块)
            """)

    # 底部信息
    gr.Markdown("---\n*Powered by VSR Enhanced | Running on Google Colab*")

# 启动
demo.launch(share=True, server_name="0.0.0.0")

## 🎉 完成！

WebUI 已启动。点击上方输出的 **public URL** 在浏览器中打开界面。

### 常见问题

| 问题 | 解决方案 |
|------|----------|
| 显存不足(OOM) | 切换到 CPU 模式或使用更小的修复算法(lama→opencv) |
| 模型下载慢 | 使用代理或镜像源 |
| 处理太慢 | 减小 detection_skip、使用 lama_fast 模式 |
| 检测不到水印 | 尝试切换检测器类型或调整 confidence_threshold |

### 输出文件位置
处理后的文件保存在 `/content/output/` 目录，也可直接通过 WebUI 下载。

---
*提示：Colab 免费版会话最长 12 小时，处理后记得下载结果文件。*